In [1]:
import numpy as np
from datetime import date

# ── Bond parameters ───────────────────────────────────────────────────────────
PARTICIPATION = 1.60      # 160%
CAP_RATE      = 0.0545    # 5.45% p.a.
FLOOR_RATE    = 0.0000    # 0.00% p.a.
NOMINAL       = 1_000     # EUR
ALPHA         = 0.25      # 30/360 quarterly accrual fraction

L_STAR     = CAP_RATE / PARTICIPATION   # 3.40625% -- cap breakeven
TRADE_DATE = date(2025, 11, 5)

In [2]:
# ── Coupon formula ────────────────────────────────────────────────────────────
def coupon_rate(L):
    """R(L) = min(cap, max(floor, participation * L))"""
    return np.minimum(CAP_RATE, np.maximum(FLOOR_RATE, PARTICIPATION * L))


def coupon_amount(L):
    """Quarterly coupon = nominal * R(L) * alpha"""
    return NOMINAL * coupon_rate(L) * ALPHA

In [3]:
# ── 30/360 day count ──────────────────────────────────────────────────────────
def days_30_360(d1: date, d2: date) -> int:
    """
    Days between d1 and d2 under the 30/360 Bond Basis convention.
    Formula: (Y2-Y1)*360 + (M2-M1)*30 + (D2-D1), with D capped at 30.
    """
    y1, m1, dd1 = d1.year, d1.month, min(d1.day, 30)
    y2, m2, dd2 = d2.year, d2.month, d2.day
    if dd2 == 31 and dd1 >= 30:
        dd2 = 30
    return (y2 - y1) * 360 + (m2 - m1) * 30 + (dd2 - dd1)

In [4]:
# ── Bond schedule ─────────────────────────────────────────────────────────────
# (period, reset_date, start_date, end_date, payment_date, euribor_fixing)
# EURIBOR fixings: Suomen Pankki EUR003M chart, accessed 5 Nov 2025.
# Payment dates unadjusted -- 12th is a T2 business day in every case.

SCHEDULE = [
    (1, date(2024,  6, 10), date(2024,  6, 12), date(2024,  9, 12), date(2024,  9, 12), 0.037430),
    (2, date(2024,  9, 10), date(2024,  9, 12), date(2024, 12, 12), date(2024, 12, 12), 0.034600),
    (3, date(2024, 12, 10), date(2024, 12, 12), date(2025,  3, 12), date(2025,  3, 12), 0.028720),
    (4, date(2025,  3, 10), date(2025,  3, 12), date(2025,  6, 12), date(2025,  6, 12), 0.025470),
    (5, date(2025,  6, 10), date(2025,  6, 12), date(2025,  9, 12), date(2025,  9, 12), 0.019540),
    (6, date(2025,  9, 10), date(2025,  9, 12), date(2025, 12, 12), date(2025, 12, 12), 0.020290),
]

In [5]:
# ── Build coupon rows ─────────────────────────────────────────────────────────
def build_coupon_table():
    rows = []
    for period, reset, start, end, payment, L in SCHEDULE:
        R      = coupon_rate(L)
        days   = days_30_360(start, end)   # 90 every period under 30/360
        alpha  = days / 360
        amount = NOMINAL * R * alpha
        rows.append({
            "period"     : period,
            "reset"      : reset,
            "start"      : start,
            "end"        : end,
            "payment"    : payment,
            "L"          : L,
            "R"          : R,
            "days"       : days,
            "alpha"      : alpha,
            "amount"     : amount,
            "cap_binding": bool(L > L_STAR),
            "paid"       : payment <= TRADE_DATE,
        })
    return rows

In [6]:
# ── Print coupon table ────────────────────────────────────────────────────────
def print_coupon_table(rows):
    w = 108
    print("\n" + "=" * w)
    print(f"  Historical Coupon Table  |  Trade date: {TRADE_DATE}  |  Nominal: EUR {NOMINAL:,}")
    print("=" * w)
    hdr = (f"  {'#':>2}  {'Reset':<12}  {'Start':<12}  {'End':<12}  "
           f"{'Payment':<12}  {'3m EURIBOR':>11}  {'Coupon rate':>12}  "
           f"{'Days':>4}  {'Coupon (EUR)':>12}  Status")
    print(hdr)
    print("-" * w)

    paid_total = 0.0
    full_total = 0.0

    for r in rows:
        flag   = " [CAP]" if r["cap_binding"] else "      "
        status = "paid" if r["paid"] else "not yet paid"
        print(
            f"  {r['period']:>2}  "
            f"{str(r['reset']):<12}  "
            f"{str(r['start']):<12}  "
            f"{str(r['end']):<12}  "
            f"{str(r['payment']):<12}  "
            f"{r['L']:>11.4%}  "
            f"{r['R']:>11.4%}{flag}  "
            f"{r['days']:>4}  "
            f"{r['amount']:>12.3f}  "
            f"{status}"
        )
        if r["paid"]:
            paid_total += r["amount"]
        full_total += r["amount"]

    print("-" * w)
    print(f"\n  {'Total coupons paid (periods 1-5)':<46} EUR {paid_total:>9.3f}")
    print(f"  {'Total incl. current fixing (periods 1-6)':<46} EUR {full_total:>9.3f}")
    print("=" * w)

    return paid_total, full_total

In [7]:
# ── Accrued interest ──────────────────────────────────────────────────────────
def print_accrued_interest(rows):
    """
    AI at trade date for period 6 (start 12 Sep 2025, trade date 5 Nov 2025).
    30/360: (11-9)*30 + (5-12) = 60 - 7 = 53 days.
    """
    r      = next(x for x in rows if x["period"] == 6)
    ai_days = days_30_360(r["start"], TRADE_DATE)
    ai      = NOMINAL * r["R"] * (ai_days / 360)

    print(f"\n  Accrued interest at trade date ({TRADE_DATE})")
    print("-" * 60)
    print(f"  Period start             : {r['start']}")
    print(f"  Trade date               : {TRADE_DATE}")
    print(f"  30/360 days accrued      : {ai_days}")
    print(f"  Current coupon rate      : {r['R']:.4%}")
    print(f"  Accrued interest (AI)    : EUR {ai:.3f}")
    print(f"  Formula: {NOMINAL} x {r['R']:.4%} x {ai_days}/360 = {ai:.3f}")
    print("-" * 60)

    return ai

In [8]:
# ── Commentary ────────────────────────────────────────────────────────────────
def print_commentary(rows, paid_total):
    cap_rows  = [r for r in rows if r["cap_binding"]]
    forgone   = sum(NOMINAL * (r["L"] * PARTICIPATION - CAP_RATE) * r["alpha"]
                    for r in cap_rows)
    plain_frn = sum(NOMINAL * r["L"] * r["alpha"] for r in rows if r["paid"])
    outperf   = paid_total - plain_frn

    print(f"\n  Commentary")
    print("-" * 60)
    print(f"  Cap binding periods       : {len(cap_rows)} "
          f"(periods {', '.join(str(r['period']) for r in cap_rows)})")
    print(f"  Floor binding periods     : 0  (EURIBOR positive throughout)")
    print(f"  EURIBOR range observed    : "
          f"{min(r['L'] for r in rows):.3%} -- {max(r['L'] for r in rows):.3%}")
    print(f"  Income forgone via cap    : EUR {forgone:.3f}")
    print()
    print(f"  Structured bond (1-5)     : EUR {paid_total:.3f}")
    print(f"  Plain FRN 100% (1-5)      : EUR {plain_frn:.3f}")
    print(f"  Outperformance            : EUR {outperf:.3f}  (participation benefit)")
    print("-" * 60)


# ── Run all sections ──────────────────────────────────────────────────────────
print("=" * 80)
print("  Q2 -- Historical Coupon Table")
print("  UniCredit Variable Rate Bond 2034  (ISIN IT0005599110)")
print("=" * 80)

rows = build_coupon_table()
paid_total, full_total = print_coupon_table(rows)
print_accrued_interest(rows)
print_commentary(rows, paid_total)

  Q2 -- Historical Coupon Table
  UniCredit Variable Rate Bond 2034  (ISIN IT0005599110)

  Historical Coupon Table  |  Trade date: 2025-11-05  |  Nominal: EUR 1,000
   #  Reset         Start         End           Payment        3m EURIBOR   Coupon rate  Days  Coupon (EUR)  Status
------------------------------------------------------------------------------------------------------------
   1  2024-06-10    2024-06-12    2024-09-12    2024-09-12        3.7430%      5.4500% [CAP]    90        13.625  paid
   2  2024-09-10    2024-09-12    2024-12-12    2024-12-12        3.4600%      5.4500% [CAP]    90        13.625  paid
   3  2024-12-10    2024-12-12    2025-03-12    2025-03-12        2.8720%      4.5952%          90        11.488  paid
   4  2025-03-10    2025-03-12    2025-06-12    2025-06-12        2.5470%      4.0752%          90        10.188  paid
   5  2025-06-10    2025-06-12    2025-09-12    2025-09-12        1.9540%      3.1264%          90         7.816  paid
   6  2025-09-